In [1]:
import pandas as pd
import numpy as np
import requests
import io

def load_and_merge_mfeat():
    base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/mfeat/"
    
    # The 6 feature views and their respective feature counts
    feature_files = {
        "mfeat-fac": 216, # Profile correlations
        "mfeat-fou": 76,  # Fourier coefficients
        "mfeat-kar": 64,  # Karhunen-Love coefficients
        "mfeat-mor": 6,   # Morphological features
        "mfeat-pix": 240, # Pixel averages
        "mfeat-zer": 47   # Zernike moments
    }
    
    dataframes = []
    
    print("Downloading and processing views...")
    for filename, count in feature_files.items():
        url = f"{base_url}{filename}"
        response = requests.get(url)
        
        if response.status_code == 200:
            # Read the file: data is space-separated with no header
            df = pd.read_csv(io.StringIO(response.text), 
                             sep=r'\s+', 
                             header=None)
            
            # Name columns based on the feature type
            df.columns = [f"{filename.split('-')[1]}_{i}" for i in range(df.shape[1])]
            dataframes.append(df)
            print(f" - Loaded {filename}: {df.shape}")
        else:
            print(f"Error downloading {filename}")

    # Concatenate all views column-wise
    full_df = pd.concat(dataframes, axis=1)

    # Generate labels: 10 digits (0-9), 200 instances each
    labels = np.repeat(np.arange(10), 200)
    full_df['digit_label'] = labels

    return full_df

# Execute the merge
mfeat_df = load_and_merge_mfeat()

print("\nMerged DataFrame Info:")
print(f"Total Instances: {len(mfeat_df)}")
print(f"Total Features (excluding label): {mfeat_df.shape[1] - 1}")
print(mfeat_df.head())

 - Loaded mfeat-fac: (2000, 216)
 - Loaded mfeat-fou: (2000, 76)
 - Loaded mfeat-kar: (2000, 64)
 - Loaded mfeat-mor: (2000, 6)
 - Loaded mfeat-pix: (2000, 240)
 - Loaded mfeat-zer: (2000, 47)

Merged DataFrame Info:
Total Instances: 2000
Total Features (excluding label): 649
   fac_0  fac_1  fac_2  fac_3  fac_4  fac_5  fac_6  fac_7  fac_8  fac_9  ...  \
0     98    236    531    673    607    647      2      9      3      6  ...   
1    121    193    607    611    585    665      7      9      2      4  ...   
2    115    141    590    605    557    627     12      6      3      3  ...   
3     90    122    627    692    607    642      0      6      4      5  ...   
4    157    167    681    666    587    666      8      6      1      4  ...   

      zer_38     zer_39    zer_40      zer_41      zer_42    zer_43  \
0  33.810340   9.858915  1.399891  148.138058  326.239452  9.711070   
1  35.400531  70.681899  6.674412  155.135985  377.832675  8.140633   
2  19.477230  30.093590  7.85